# Music Annotation Project V7.8.3: Read-Only Frozen-Policy Application

**Runtime:** CPU. Select **Runtime > Run all**.

This notebook applies one already frozen V7.7F structural-safety policy to one already released and locked V7.8 `component_dataset.csv`.

## Hard rules

- Do not refit the structural model.
- Do not refit the median imputer.
- Do not refit or replace the isotonic calibrator.
- Do not change the ordered predictor schema.
- Do not change the threshold.
- Do not use V7.8 structural outcomes until after frozen scores and decisions have been calculated.
- Do not overwrite any V7.7F or V7.8 source artefact.

The notebook verifies release, lock, model, calibrator, schema and replay hashes before scoring. It writes all results to a new timestamped directory.

## Verdicts

1. **Integrity verdict**: whether the exact frozen policy and exact locked V7.8 release were loaded and applied without mutation.
2. **Independent evidence verdict**: observed false-safe performance, coverage and uncertainty.
3. **Production decision**: produced only when acceptance criteria were explicitly configured before the run. Otherwise the result is `EVIDENCE_ONLY_NOT_PRODUCTION_AUTHORISED`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy scikit-learn joblib matplotlib


In [ ]:
import json, hashlib, math, platform, sys
import numpy as np
import pandas as pd
import sklearn, joblib
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
POLICY_FREEZE_ROOT=ROOT/'structural_policy_freeze_v7_7'
V78_ROOT_CANDIDATES=[
 ROOT/'independent_structural_validation_v7_8_rebuilt_telea_immutable',
 ROOT/'independent_structural_validation_v7_8_rebuilt',
]
OUTPUT_ROOT=ROOT/'independent_policy_validation_v7_8'

# Prefer explicit paths for the formal run. None means deterministic discovery of the newest complete candidate.
POLICY_RUN_OVERRIDE=None
V78_RELEASE_OVERRIDE=None

# Optional predeclared production acceptance criteria.
# Leave either as None to produce evidence only, without production authorisation.
MAX_FALSE_SAFE_WILSON95_UPPER=None   # Example only: 0.01 must not be adopted retrospectively.
MIN_SELECTED_COMPONENTS=None         # Example only: 100 must not be adopted retrospectively.
REQUIRE_ALL_SOURCE_GROUPS_SELECTED=False
REQUIRE_ZERO_OBSERVED_FALSE_SAFE=True

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_frozen_policy_application'
OUT=OUTPUT_ROOT/RUN;TABLES=OUT/'tables';CHARTS=OUT/'charts';FAILURES=OUT/'false_safe_records'
for d in [OUT,TABLES,CHARTS,FAILURES]:d.mkdir(parents=True,exist_ok=False)

def sha256_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for block in iter(lambda:f.read(1048576),b''):h.update(block)
 return h.hexdigest()
def atomic_json(path,value):
 tmp=Path(str(path)+'.tmp');tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str));tmp.replace(path)
def canonical_hash(value):return hashlib.sha256(json.dumps(value,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def wilson_upper(errors,n,z=1.96):
 if n==0:return None
 p=errors/n;den=1+z*z/n;centre=p+z*z/(2*n);radius=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n));return float((centre+radius)/den)
def newest_complete(root,required):
 hits=[]
 if root.exists():
  for p in root.rglob('*'):
   if p.is_dir() and all((p/r).exists() for r in required):hits.append(p)
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None


## Discover the exact policy and V7.8 release


In [ ]:
POLICY_REQUIRED=['artefacts/policy.json','artefacts/predictor_schema.json','artefacts/structural_model.joblib','artefacts/isotonic_calibrator.joblib','artefacts/replay_test.json','v7_7_policy_freeze_manifest.json']
POLICY_RUN=Path(POLICY_RUN_OVERRIDE) if POLICY_RUN_OVERRIDE else newest_complete(POLICY_FREEZE_ROOT,POLICY_REQUIRED)
if POLICY_RUN is None:raise RuntimeError('No complete V7.7F policy-freeze run found. Set POLICY_RUN_OVERRIDE.')

V78_REQUIRED=['component_dataset.csv','dataset_release.json','score_manifest.csv']
if V78_RELEASE_OVERRIDE:
 V78=Path(V78_RELEASE_OVERRIDE)
else:
 candidates=[p for p in V78_ROOT_CANDIDATES if all((p/r).exists() for r in V78_REQUIRED)]
 if not candidates:raise RuntimeError('No complete released V7.8 dataset found. Set V78_RELEASE_OVERRIDE.')
 V78=max(candidates,key=lambda p:(p/'dataset_release.json').stat().st_mtime)

POLICY_DIR=POLICY_RUN/'artefacts';POLICY_PATH=POLICY_DIR/'policy.json';SCHEMA_PATH=POLICY_DIR/'predictor_schema.json';MODEL_PATH=POLICY_DIR/'structural_model.joblib';CALIBRATOR_PATH=POLICY_DIR/'isotonic_calibrator.joblib';REPLAY_PATH=POLICY_DIR/'replay_test.json'
COMPONENT_PATH=V78/'component_dataset.csv';RELEASE_PATH=V78/'dataset_release.json';SCORE_MANIFEST_PATH=V78/'score_manifest.csv'
policy=json.loads(POLICY_PATH.read_text());schema=json.loads(SCHEMA_PATH.read_text());replay=json.loads(REPLAY_PATH.read_text());release=json.loads(RELEASE_PATH.read_text())
LOCK_ID=str(release.get('lock_id',''));LOCK_PATH=V78/'locks'/f'{LOCK_ID}.json'
print('Policy run:',POLICY_RUN);print('Policy ID:',policy.get('policy_id'));print('V7.8 release:',V78);print('Lock ID:',LOCK_ID);print('Output:',OUT)


## Fail-closed provenance and hash gate


In [ ]:
checks={}
checks['policy_candidate_only']=policy.get('candidate_policy_only') is True
checks['policy_requires_independent_validation']=policy.get('independent_validation_required') is True
checks['policy_replay_passed']=replay.get('passed') is True and replay.get('policy_id')==policy.get('policy_id')
checks['model_hash']=policy.get('model_sha256')==sha256_file(MODEL_PATH)
checks['calibrator_hash']=policy.get('calibrator_sha256')==sha256_file(CALIBRATOR_PATH)
checks['schema_hash']=policy.get('schema_sha256')==sha256_file(SCHEMA_PATH)
checks['component_release_hash']=release.get('component_dataset_sha256')==sha256_file(COMPONENT_PATH)
checks['score_manifest_release_hash']=release.get('score_manifest_sha256')==sha256_file(SCORE_MANIFEST_PATH)
checks['release_lock_id_present']=bool(LOCK_ID)
checks['lock_file_exists']=LOCK_PATH.exists()
lock=json.loads(LOCK_PATH.read_text()) if LOCK_PATH.exists() else {}
checks['lock_id_matches_file']=lock.get('lock_id')==LOCK_ID
checks['lock_status_pre_outcome']=lock.get('status')=='LOCKED_BEFORE_STRUCTURAL_OUTCOMES'
checks['deterministic_lock']=lock.get('deterministic_identity') is True
payload=lock.get('payload',{})
checks['released_component_count_matches_lock']=True  # completed after dataset load
checks['released_page_count_matches_lock']=True
checks['released_source_group_count_matches_lock']=True
if not all(v for k,v in checks.items() if k not in ['released_component_count_matches_lock','released_page_count_matches_lock','released_source_group_count_matches_lock']):
 atomic_json(OUT/'failed_integrity_checks.json',checks);raise RuntimeError(f'INTEGRITY GATE FAILED: {checks}')
print('Static policy/release hash checks passed.')


## Load the locked dataset and enforce the frozen schema


In [ ]:
df=pd.read_csv(COMPONENT_PATH);score_manifest=pd.read_csv(SCORE_MANIFEST_PATH);predictors=list(schema.get('ordered_predictors',[]))
required_truth={'source_group','page','component','structural_safe','structural_failure_reasons'}
if not required_truth.issubset(df.columns):raise RuntimeError(f'V7.8 component dataset missing: {sorted(required_truth-set(df.columns))}')
if not predictors:raise RuntimeError('Frozen predictor schema is empty.')
missing=[c for c in predictors if c not in df.columns]
if missing:raise RuntimeError(f'Missing frozen predictors: {missing}')
if df.duplicated(['page','component']).any():raise RuntimeError('Duplicate V7.8 page/component rows.')
if not set(df.page.astype(str)).issubset(set(score_manifest.page.astype(str))):raise RuntimeError('Component pages are not all present in score_manifest.csv.')

X=df.loc[:,predictors].copy()
for c in predictors:X[c]=pd.to_numeric(X[c],errors='coerce')
arr=X.to_numpy(dtype=float)
if np.isinf(arr).any():raise RuntimeError('Infinite predictor value found.')
# NaN is allowed only because the serialised pipeline contains the already-fitted median imputer.

checks['released_component_count_matches_lock']=int(payload.get('components',-1))==len(df)
checks['released_page_count_matches_lock']=int(payload.get('pages',-1))==df.page.nunique()
checks['released_source_group_count_matches_lock']=int(payload.get('source_groups',-1))==df.source_group.nunique()
if not all(checks.values()):
 atomic_json(OUT/'failed_integrity_checks.json',checks);raise RuntimeError(f'LOCK/DATASET COUNT GATE FAILED: {checks}')
print('Dataset rows:',len(df),'Pages:',df.page.nunique(),'Source groups:',df.source_group.nunique(),'Predictors:',len(predictors))


## Score predictors before reading structural truth


In [ ]:
# The model calculation uses X only. Structural outcomes are not passed to either artefact.
model=joblib.load(MODEL_PATH);calibrator=joblib.load(CALIBRATOR_PATH)
raw_score=model.predict_proba(X)[:,1]
probability=calibrator.predict(raw_score)
threshold=float(policy['threshold'])
selected=probability>=threshold
if not np.isfinite(raw_score).all() or not np.isfinite(probability).all():raise RuntimeError('Non-finite frozen-policy score produced.')
if len(raw_score)!=len(df):raise RuntimeError('Score row count mismatch.')

scored=df.copy()
scored['raw_structural_safe_score']=raw_score
scored['prob_structural_safe']=probability
scored['candidate_structural_auto_safe']=selected
scored['policy_threshold']=threshold
scored['policy_id']=policy['policy_id']
SCORED_PATH=TABLES/'v7_8_policy_scored_components.csv';scored.to_csv(SCORED_PATH,index=False)
score_identity={'policy_id':policy['policy_id'],'v7_8_lock_id':LOCK_ID,'component_dataset_sha256':sha256_file(COMPONENT_PATH),'ordered_predictors':predictors,'threshold':threshold,'raw_score_hash':canonical_hash([float(x) for x in raw_score]),'probability_hash':canonical_hash([float(x) for x in probability]),'decision_hash':canonical_hash([bool(x) for x in selected])}
atomic_json(OUT/'frozen_score_identity.json',score_identity)
print('Frozen scoring complete. Threshold:',threshold,'Selected:',int(selected.sum()))


## Reveal independent outcomes and calculate false-safe evidence


In [ ]:
def as_bool(series):
 if pd.api.types.is_bool_dtype(series):return series.to_numpy(dtype=bool)
 values=series.astype(str).str.strip().str.lower()
 if not values.isin(['true','false','1','0','yes','no']).all():raise RuntimeError('structural_safe contains invalid Boolean values.')
 return values.isin(['true','1','yes']).to_numpy()
truth=as_bool(scored.structural_safe)
false_safe=selected&~truth;true_safe_selected=selected&truth;safe_not_selected=~selected&truth
selected_count=int(selected.sum());false_safe_count=int(false_safe.sum());safe_total=int(truth.sum())
headline={'independent_components':len(scored),'independent_pages':int(scored.page.nunique()),'independent_source_groups':int(scored.source_group.nunique()),'structural_safe_components':safe_total,'structural_unsafe_components':int((~truth).sum()),'frozen_policy_selected':selected_count,'true_safe_selected':int(true_safe_selected.sum()),'false_safe_selected':false_safe_count,'safe_not_selected':int(safe_not_selected.sum()),'selection_coverage':selected_count/len(scored) if len(scored) else None,'observed_false_safe_rate':false_safe_count/selected_count if selected_count else None,'false_safe_rate_wilson95_upper':wilson_upper(false_safe_count,selected_count),'structural_safe_recall':int(true_safe_selected.sum())/safe_total if safe_total else None}
print(json.dumps(headline,indent=2))


## Source-group, page and failure-reason breakdowns


In [ ]:
def group_summary(g):
 sel=g.candidate_structural_auto_safe.astype(bool).to_numpy();tru=as_bool(g.structural_safe);errors=int(np.sum(sel&~tru));n=int(sel.sum())
 return pd.Series({'components':len(g),'pages':g.page.nunique(),'structural_safe':int(tru.sum()),'structural_unsafe':int((~tru).sum()),'selected':n,'false_safe':errors,'observed_false_safe_rate':errors/n if n else np.nan,'false_safe_wilson95_upper':wilson_upper(errors,n),'selection_coverage':n/len(g) if len(g) else np.nan,'structural_safe_recall':int(np.sum(sel&tru))/max(int(tru.sum()),1)})
by_source=scored.groupby('source_group',dropna=False).apply(group_summary,include_groups=False).reset_index()
by_page=scored.groupby(['source_group','page'],dropna=False).apply(group_summary,include_groups=False).reset_index()
by_source.to_csv(TABLES/'policy_results_by_source_group.csv',index=False);by_page.to_csv(TABLES/'policy_results_by_page.csv',index=False)

false_safe_df=scored.loc[false_safe].copy();safe_not_selected_df=scored.loc[safe_not_selected].copy()
false_safe_df.to_csv(TABLES/'false_safe_components.csv',index=False);safe_not_selected_df.to_csv(TABLES/'safe_not_selected_components.csv',index=False)
reason_rows=[]
for _,r in false_safe_df.iterrows():
 reasons=[x for x in str(r.structural_failure_reasons).split('|') if x and x.lower()!='nan']
 if not reasons:reasons=['unspecified']
 for reason in reasons:reason_rows.append({'source_group':r.source_group,'page':r.page,'component':r.component,'reason':reason,'prob_structural_safe':r.prob_structural_safe})
reason_df=pd.DataFrame(reason_rows,columns=['source_group','page','component','reason','prob_structural_safe'])
reason_summary=(reason_df.groupby('reason',as_index=False).agg(false_safe_components=('component','size'),pages=('page','nunique'),source_groups=('source_group','nunique')).sort_values('false_safe_components',ascending=False) if len(reason_df) else pd.DataFrame(columns=['reason','false_safe_components','pages','source_groups']))
reason_df.to_csv(TABLES/'false_safe_failure_reasons_long.csv',index=False);reason_summary.to_csv(TABLES/'false_safe_failure_reason_summary.csv',index=False)
display(by_source);display(false_safe_df[['source_group','page','component','prob_structural_safe','structural_failure_reasons']].head(100))


## Charts


In [ ]:
plt.figure(figsize=(9,5));q=by_source.sort_values('selection_coverage');plt.barh(q.source_group,q.selection_coverage);plt.xlabel('Selection coverage');plt.title('Frozen structural policy coverage by independent source group');plt.tight_layout();plt.savefig(CHARTS/'selection_coverage_by_source_group.png',dpi=180);plt.close()
plt.figure(figsize=(9,5));q=by_source.sort_values('false_safe_wilson95_upper');plt.barh(q.source_group,q.false_safe_wilson95_upper.fillna(0));plt.xlabel('Wilson 95% upper bound on false-safe rate');plt.title('Independent structural-risk uncertainty by source group');plt.tight_layout();plt.savefig(CHARTS/'false_safe_wilson_upper_by_source_group.png',dpi=180);plt.close()
plt.figure(figsize=(9,5));plt.hist(probability,bins=30);plt.axvline(threshold,color='red',linestyle='--',label=f'Frozen threshold {threshold:.4f}');plt.xlabel('Frozen calibrated structural-safe probability');plt.ylabel('Components');plt.title('V7.8 independent frozen-policy scores');plt.legend();plt.tight_layout();plt.savefig(CHARTS/'independent_probability_distribution.png',dpi=180);plt.close()


## Fail-closed validation report


In [ ]:
criteria_configured=MAX_FALSE_SAFE_WILSON95_UPPER is not None and MIN_SELECTED_COMPONENTS is not None
criteria_checks={
 'minimum_selected_components':None if MIN_SELECTED_COMPONENTS is None else selected_count>=int(MIN_SELECTED_COMPONENTS),
 'maximum_false_safe_wilson95_upper':None if MAX_FALSE_SAFE_WILSON95_UPPER is None or headline['false_safe_rate_wilson95_upper'] is None else headline['false_safe_rate_wilson95_upper']<=float(MAX_FALSE_SAFE_WILSON95_UPPER),
 'zero_observed_false_safe':(false_safe_count==0) if REQUIRE_ZERO_OBSERVED_FALSE_SAFE else True,
 'all_source_groups_selected':(by_source.selected>0).all() if REQUIRE_ALL_SOURCE_GROUPS_SELECTED else True,
}
if not criteria_configured:
 production_verdict='EVIDENCE_ONLY_NOT_PRODUCTION_AUTHORISED'
elif selected_count==0:
 production_verdict='FAIL_NO_COMPONENTS_SELECTED'
elif all(v is True for v in criteria_checks.values()):
 production_verdict='PASS_PREDECLARED_STRUCTURAL_POLICY_CRITERIA'
else:
 production_verdict='FAIL_PREDECLARED_STRUCTURAL_POLICY_CRITERIA'

integrity_verdict='PASS' if all(checks.values()) else 'FAIL'
report={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'7.8.3','read_only':True,'refit_performed':False,'recalibration_performed':False,'threshold_changed':False,'integrity_verdict':integrity_verdict,'production_verdict':production_verdict,'policy_id':policy['policy_id'],'policy_threshold':threshold,'v7_8_lock_id':LOCK_ID,'source_policy_run':str(POLICY_RUN),'source_v7_8_release':str(V78),'hash_checks':checks,'headline':headline,'acceptance_criteria':{'configured_before_run':criteria_configured,'maximum_false_safe_wilson95_upper':MAX_FALSE_SAFE_WILSON95_UPPER,'minimum_selected_components':MIN_SELECTED_COMPONENTS,'require_zero_observed_false_safe':REQUIRE_ZERO_OBSERVED_FALSE_SAFE,'require_all_source_groups_selected':REQUIRE_ALL_SOURCE_GROUPS_SELECTED},'acceptance_checks':criteria_checks,'software':{'python':sys.version,'platform':platform.platform(),'numpy':np.__version__,'pandas':pd.__version__,'scikit_learn':sklearn.__version__,'joblib':joblib.__version__},'output_hashes':{}}
for p in sorted(TABLES.glob('*.csv')):report['output_hashes'][str(p.relative_to(OUT))]=sha256_file(p)
report['output_hashes'][str((OUT/'frozen_score_identity.json').relative_to(OUT))]=sha256_file(OUT/'frozen_score_identity.json')
atomic_json(OUT/'independent_policy_summary.json',report)

lines=['V7.8.3 READ-ONLY FROZEN STRUCTURAL-POLICY APPLICATION','',f'Integrity verdict: {integrity_verdict}',f'Production verdict: {production_verdict}',f'Policy ID: {policy["policy_id"]}',f'V7.8 lock ID: {LOCK_ID}',f'Frozen threshold: {threshold}',f'Independent components: {len(scored)}',f'Selected: {selected_count}',f'False-safe selected: {false_safe_count}',f'Observed false-safe rate: {headline["observed_false_safe_rate"]}',f'Wilson 95% upper bound: {headline["false_safe_rate_wilson95_upper"]}',f'Selection coverage: {headline["selection_coverage"]}',f'Structural-safe recall: {headline["structural_safe_recall"]}','','No refit, recalibration or threshold change was performed.']
if not criteria_configured:lines+=['','No complete predeclared acceptance criteria were configured. This run is evidence only and does not authorise production auto-acceptance.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines));print(chr(10).join(lines));print('Output:',OUT)
if integrity_verdict!='PASS':raise RuntimeError('FAIL-CLOSED: integrity verdict failed.')
